In [97]:
from dbfread import DBF
import pandas as pd
import numpy as np

## Data preparation
### Raw data ingestion

In [98]:
def read_dbf(file_path):
	# read the raw data and store in a dataframe
	dbf = DBF(file_path)
	df = pd.DataFrame(iter(dbf))

	# identify empty strings as missing values
	df.replace("", np.nan, inplace=True)

	# ensure there are now empty rows or columns
	df.dropna(how='all', axis=0, inplace=True)
	df.dropna(how='all', axis=1, inplace=True)

	# remove any duplicates
	df.drop_duplicates(inplace=True)

	# standardize column names
	df.columns = map(lambda x: x.lower(), df.columns)

	return df

#### Expedition data

In [99]:
exped_df = read_dbf('data/raw/exped.DBF')

In [100]:
exped_df.shape

(11578, 66)

In [101]:
exped_df.head()

,expid,peakid,year,season,host,route1,route2,route3,route4,nation,...,accidents,achievment,agency,comrte,stdrte,primrte,primmem,primref,primid,chksum
0,ANN260101,ANN2,1960,1,1,NW Ridge-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2442047
1,ANN269301,ANN2,1969,3,1,NW Ridge-W Ridge,NaN,NaN,NaN,Yugoslavia,...,Draslar frostbitten hands and feet,NaN,NaN,None,None,False,False,None,NaN,2445501
2,ANN273101,ANN2,1973,1,1,W Ridge-N Face,NaN,NaN,NaN,Japan,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2446797
3,ANN278301,ANN2,1978,3,1,N Face-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2448822
4,ANN279301,ANN2,1979,3,1,N Face-W Ridge,NW Ridge of A-IV,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2449204


### Data Cleaning

In [102]:
exped_df.groupby('expid').expid.count().sort_values(ascending=False)[:1]

expid
KANG10101    2
Name: expid, dtype: int64

In [103]:
exped_df.loc[exped_df.expid == 'KANG10101']

,expid,peakid,year,season,host,route1,route2,route3,route4,nation,...,accidents,achievment,agency,comrte,stdrte,primrte,primmem,primref,primid,chksum
2860,KANG10101,KANG,1910,1,3,NE Side (recon),NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2405
6780,KANG10101,KANG,2010,1,1,SW Face,NaN,NaN,NaN,S Korea,...,NaN,NaN,Windhorse Trekking,False,True,False,False,False,NaN,2457821


> Some _expid_ values are duplicated for expeditions to the same peak occuring one century appart

In [104]:
# add the full year to the expedition id to ensure uniqueness
exped_df.expid = exped_df.expid.str.cat(exped_df.year)
assert exped_df.expid.nunique() == exped_df.shape[0]

In [105]:
# map country index to name as per the documentation
host_map = {
	0: 'Unknown',
	1: 'Nepal',
	2: 'China',
	3: 'India'
}

exped_df.host = exped_df.host.map(host_map)

In [106]:
# map season index to name as per the documentation
season_map = {
	0: 'Unknown',
	1: 'Spring',
	2: 'Summer',
	3: 'Autumn',
	4: 'Winter'
}

exped_df.season = exped_df.season.map(season_map)

In [107]:
# remove expedition with undefined main route
exped_df = exped_df.loc[exped_df.route1.notna()]

In [108]:
# remove expeditions that include non-climbing activities
exped_df = exped_df.loc[~exped_df.traverse & ~exped_df.ski & ~exped_df.parapente]

exped_df.drop(['traverse', 'ski', 'parapente'], axis=1, inplace=True)

In [109]:
# filter based on expedition termination reason:
# 12 - Did not attempt climb
# 13 - Attempt rumored
exped_df = exped_df.loc[
	~exped_df.termreason.isin([12, 13])
]

exped_df.drop('termreason', axis=1, inplace=True)

In [110]:
# remove unused columns
exped_df.drop([
	'route2', 'route3', 'route4', 'success2', 'success3', 'success4', 'ascent2', 'ascent3', 'ascent4', 'claimed',
	'disputed', 'approach', 'smtdate', 'smttime', 'smtdays', 'totdays', 'termdate', 'termnote', 'highpoint',
	'smtmembers', 'mdeaths', 'smthired', 'hdeaths', 'othersmts', 'campsites', 'routememo', 'accidents', 'achievment',
	'primmem', 'primref', 'primid', 'chksum', 'leaders', 'countries', 'ascent1', 'bcdate', 'o2used', 'o2none',
	'o2medical', 'o2unkwn', 'agency', 'o2taken', 'nohired', 'rope'], axis=1, inplace=True)

In [111]:
exped_df.shape

(10888, 18)

> After initial cleaning and filtering, we are left with a dataset of 10,888 expeditions

In [112]:
# create flag variables to indicate whether the expedition has a sponsor
exped_df['sponsored'] = exped_df.sponsor.notna()
exped_df.drop('sponsor', axis=1, inplace=True)

In [113]:
# concatenate peak and route
exped_df['ascent_route'] = exped_df.peakid.str.cat(exped_df.route1, sep='-').str.replace(" ", "_")
exped_df.drop(['peakid', 'route1'], axis=1, inplace=True)

In [114]:
# find most commonly attempted ascents
route_counts = pd.DataFrame(exped_df.ascent_route.value_counts()).reset_index()
common_routes = route_counts.loc[route_counts['count'] >= 10, 'ascent_route']

In [115]:
# keep only expeditions on common routes
exped_df = exped_df.loc[exped_df.ascent_route.isin(common_routes)]

In [116]:
exped_df.shape

(7900, 17)

In [117]:
exped_df.head()

,expid,year,season,host,nation,success1,camps,totmembers,tothired,o2climb,o2descent,o2sleep,comrte,stdrte,primrte,sponsored,ascent_route
21,ANN4502011950,1950,Summer,Nepal,UK,False,4,6,5,False,False,False,None,None,False,True,ANN4-NW_Ridge
25,ANN4551011955,1955,Spring,Nepal,W Germany,True,4,4,2,False,False,False,None,None,False,True,ANN4-NW_Ridge
26,ANN4571011957,1957,Spring,Nepal,UK,True,4,2,4,False,False,False,None,None,False,False,ANN4-NW_Ridge
27,ANN4601011960,1960,Spring,Nepal,UK,True,5,4,3,False,False,False,None,None,False,False,ANN4-NW_Ridge
28,ANN4693011969,1969,Autumn,Nepal,Czechoslovakia,True,4,9,1,False,False,False,None,None,False,True,ANN4-NW_Ridge


In [118]:
exped_df.success1.value_counts()

success1
True     4893
False    3007
Name: count, dtype: int64

#### Climber data

In [119]:
climber_df = read_dbf('data/raw/members.DBF')

In [120]:
climber_df.head()

,expid,membid,peakid,myear,mseason,fname,lname,sex,age,yob,...,membermemo,necrology,msmtbid,msmtterm,hcn,mchksum,msmtnote1,msmtnote2,msmtnote3,deathrte
0,AMAD78301,01,AMAD,1978,3,Jean Robert,Clemenson,M,0,1938,...,None,None,1,4,0,2426937,NaN,NaN,NaN,NaN
1,AMAD78301,02,AMAD,1978,3,Bernard,Dufour,M,0,1936,...,None,None,1,4,0,2426501,NaN,NaN,NaN,NaN
2,AMAD78301,03,AMAD,1978,3,Philippe,Gerard,M,0,1950,...,None,None,1,4,0,2431569,NaN,NaN,NaN,NaN
3,AMAD78301,04,AMAD,1978,3,Eric,Lasserre,M,0,1937,...,None,None,1,4,0,2426809,NaN,NaN,NaN,NaN
4,AMAD78301,05,AMAD,1978,3,Guy,Peters,M,0,1944,...,None,None,1,4,0,2429215,NaN,NaN,NaN,NaN


In [121]:
climber_df = climber_df[[
	'expid', 'myear', 'mseason', 'fname', 'lname', 'yob', 'status', 'leader', 'support', 'disabled', 'hired', 'sherpa',
	'tibetan', 'msuccess']].dropna(how='any', subset=['expid', 'myear', 'fname', 'lname', 'yob'])

In [122]:
climber_df.expid = climber_df.expid.str.cat(climber_df.myear)

In [123]:
climber_df.sort_values(['fname', 'lname', 'yob', 'myear', 'mseason'], inplace=True)

In [125]:
climber_df['experience'] = climber_df.groupby(['fname', 'lname', 'yob'], as_index=False).expid.cumcount()
climber_df['leadership_experience'] = climber_df.groupby(['fname', 'lname', 'yob'], as_index=False).leader.cumsum()
climber_df['successes'] = climber_df.groupby(['fname', 'lname', 'yob'], as_index=False).msuccess.cumsum()

In [128]:
climber_df.head(10)

,expid,myear,mseason,fname,lname,yob,status,leader,support,disabled,hired,sherpa,tibetan,msuccess,experience,leadership_experience,successes
25302,AMAD993071999,1999,3,(William) Bradley,Singley,1966,Climber,False,False,False,False,False,False,False,0,0,0
70697,LDAK173012017,2017,3,A.,Mascini,1989,Climber,False,False,False,False,False,False,True,0,0,1
26452,EVER601011960,1960,1,A. B. (Jungi),Jungalwala,1933,Climber,False,False,False,False,False,False,False,0,0,0
26504,ANN3611011961,1961,1,A. B. (Jungi),Jungalwala,1933,Climber,False,False,False,False,False,False,False,1,0,0
26561,EVER621011962,1962,1,A. B. (Jungi),Jungalwala,1933,Climber,False,False,False,False,False,False,False,2,0,0
8601,EVER753011975,1975,3,A. Christopher (Chris),Ralling,1929,Climber,False,False,False,False,False,False,False,0,0,0
27330,CHOY001132000,2000,1,A. J.,La Fleur,1947,Climber,False,False,False,False,False,False,False,0,0,0
19593,NUPT751011975,1975,1,A. J. (John),Muston,1934,Climber,False,False,False,False,False,False,False,0,0,0
8631,EVER761011976,1976,1,A. J. (John),Muston,1934,Climber,False,False,False,False,False,False,False,1,0,0
3351,APIM801011980,1980,1,A. J. N. (Andy),Simkins,1952,Climber,False,False,False,False,False,False,False,0,0,0
